# LIBERO-PRO, Object suite, under the papers' protocol
**VLA direction** · LIBERO-PRO (arXiv 2510.03827) · released starVLA checkpoint `Qwen3-VL-PI-LIBERO-4in1`, VLM frozen · 14 Nov 2026 · 50 start states per task, three evaluation seeds

- **The question:** the best configuration of 13 Nov solved 133 and 136 of 200 rollouts without any oracle plan, on 20 start states per task and one seed. Does it hold on all 50 start states the papers use, over several seeds, and where does it stand next to the published numbers?
- **Real:** the benchmark's scenes, start states and goal checks; the released policy; the action head trained with the labels of 13 Nov on all ten LIBERO-Object tasks (`s9_align_release10`); the plan head that scores the image cells of the agent view (`g1`) and its plans, each kept until the gripper has closed or opened; all rollouts.
- **ORACLE:** the plan of the third configuration (object position from the simulator plus the average grasp point of the demonstrations), an upper bound for the plan head. The plan head turns a pixel and a depth into a 3D point with the simulator's camera calibration.
- **Protocol:** 10 tasks × 50 start states × 2 perturbations = 1000 rollouts per configuration and seed; seeds 7, 8 and 9 (the seed of the policy's sampling noise and of the simulator). All policies saw only standard LIBERO demonstrations. One trained model per configuration: the seeds vary the evaluation, not the training.

```text
(1) success per seed, mean and spread                      real rollouts
        ↓
(2) next to the numbers other papers report                 quoted, not re-run
        ↓
(3) per task, pooled over the seeds                         real
        ↓
(4) what happened to the named object                       real
```

<sub>To re-run on the A6000 server, through the GPU queue: `scripts/eval/run_libero_pro_gpu.sh` with `TRIALS=50 SEED=<seed>` and `NAME=full_released_seed<seed>`; `NAME=full_g1_hold_s9_seed<seed> ADAPTER=s9_align_release10 PLAN=predicted PREDICTOR=g1 HOLD=1`; `NAME=full_oracle_s9_seed<seed> ADAPTER=s9_align_release10 PLAN=oracle`. Then execute this notebook.</sub>

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "affordance_vla").is_dir())
sys.path.insert(0, str(ROOT))
from report import libero_pro_helpers as L

pd.set_option("display.max_colwidth", 120)
PRO = ROOT / "runs" / "libero_pro"
SEEDS = (7, 8, 9)
EXPECTED = 500   # rollouts per perturbation and seed: 10 tasks x 50 start states
RELEASED = "released policy, no plan"
MAP = "ten-task head of 13 Nov + map plans, kept"
ORACLE = "ten-task head of 13 Nov + ORACLE plan"
CONFIGS = {RELEASED: "full_released_seed{}", MAP: "full_g1_hold_s9_seed{}", ORACLE: "full_oracle_s9_seed{}"}


def load(folder):
    has_logs = lambda suite: (folder / suite).is_dir() and any((folder / suite).glob("task*_ep*.npz"))  # noqa: E731
    return {suite: L.load_suite(folder, suite) if has_logs(suite) else [] for suite in L.SUITES.values()}


runs = {name: {seed: load(PRO / pattern.format(seed)) for seed in SEEDS} for name, pattern in CONFIGS.items()}
counts = {name: {seed: [len(logs) for logs in run.values()] for seed, run in seeds.items()}
          for name, seeds in runs.items()}
print("rollouts per configuration, seed and perturbation:", counts)

rollouts per configuration, seed and perturbation: {'released policy, no plan': {7: [500, 500], 8: [500, 500], 9: [500, 500]}, 'ten-task head of 13 Nov + map plans, kept': {7: [500, 500], 8: [500, 500], 9: [500, 500]}, 'ten-task head of 13 Nov + ORACLE plan': {7: [500, 500], 8: [500, 500], 9: [500, 500]}}


## 1 · Success per seed
**This step:** rollouts in which the benchmark's goal check held, for each seed, and the mean and standard deviation of the success rate over the complete seeds (500 rollouts each).

**Status:** real rollouts. The third configuration is given the ORACLE plan.

In [2]:
table = L.seed_table(runs, expected=EXPECTED)
display(table.style.format({"mean": "{:.3f}", "std over seeds": "{:.3f}"}, na_rep="–"))

## 2 · Next to the published numbers
**This step:** the means of section 1 next to the numbers other papers report for the same two perturbations of the Object suite.

**Status:** the published numbers are quoted, not re-run. The papers use 50 start states per task too; how many seeds they ran, and which instruction they fed under the task perturbation, is not always stated. GAM uses the simulator's depth; the affordance-field method uses depth and GPT-4o at test time. Three papers report three different values for π0.5 under the position perturbation.

In [3]:
rows = {name: {"position": values[1], "task": tasks[1], "what it is": what}
        for name, what, values, tasks in L.PUBLISHED}
for name in CONFIGS:
    rows[f"ours: {name}"] = {"what it is": "this notebook, mean over complete seeds",
                             "position": table.loc[(name, "position"), "mean"],
                             "task": table.loc[(name, "task"), "mean"]}
display(pd.DataFrame.from_dict(rows, orient="index")[["what it is", "position", "task"]]
        .style.format({"position": "{:.2f}", "task": "{:.2f}"}, na_rep="–"))

,what it is,position,task
"π0.5, as reported in the GAM paper",official checkpoint,0.17,0.01
"π0.5, as reported in the CounterAlign paper",trained on LIBERO,0.19,0.10
GAM (arXiv 2609.23863),"not a VLA; 3D detector, simulator depth",0.47,0.50
CounterAlign (arXiv 2608.21740),π0.5 + offline RL,0.51,0.26
π0.5 + Affordance Field Intervention (arXiv 2512.07472),test-time depth and GPT-4o,0.73,–
"ours: released policy, no plan","this notebook, mean over complete seeds",0.00,0.09
"ours: ten-task head of 13 Nov + map plans, kept","this notebook, mean over complete seeds",0.66,0.68
ours: ten-task head of 13 Nov + ORACLE plan,"this notebook, mean over complete seeds",0.92,0.75


## 3 · Per task, pooled over the seeds
**This step:** for each task, the solved rollouts of each configuration over all seeds (150 per task when all seeds are complete).

**Status:** real.

In [4]:
pooled = {name: {suite: [log for run in seeds.values() for log in run[suite]] for suite in L.SUITES.values()}
          for name, seeds in runs.items()}
for label, suite in L.SUITES.items():
    print(label)
    display(L.per_task_table(pooled, suite))

position: the target swaps places with another object


,instruction given,"released policy, no plan","ten-task head of 13 Nov + map plans, kept",ten-task head of 13 Nov + ORACLE plan
task,,,,
0,pick up the alphabet soup and place it in the basket,0 / 150,124 / 150,141 / 150
1,pick up the cream cheese and place it in the basket,0 / 150,147 / 150,150 / 150
2,pick up the salad dressing and place it in the basket,0 / 150,100 / 150,124 / 150
3,pick up the bbq sauce and place it in the basket,0 / 150,78 / 150,149 / 150
4,pick up the ketchup and place it in the basket,0 / 150,56 / 150,104 / 150
5,pick up the tomato sauce and place it in the basket,0 / 150,92 / 150,142 / 150
6,pick up the butter and place it in the basket,0 / 150,71 / 150,150 / 150
7,pick up the milk and place it in the basket,0 / 150,108 / 150,150 / 150
8,pick up the chocolate pudding and place it in the basket,0 / 150,141 / 150,149 / 150


task: the instruction and the goal name another object


,instruction given,"released policy, no plan","ten-task head of 13 Nov + map plans, kept",ten-task head of 13 Nov + ORACLE plan
task,,,,
0,pick up the cream cheese and place it in the basket,0 / 150,146 / 150,150 / 150
1,pick up the alphabet soup and place it in the basket,135 / 150,142 / 150,145 / 150
2,pick up the tomato sauce and place it in the basket,0 / 150,106 / 150,131 / 150
3,pick up the ketchup and place it in the basket,0 / 150,7 / 150,31 / 150
4,pick up the milk and place it in the basket,0 / 150,24 / 150,24 / 150
5,pick up the bbq sauce and place it in the basket,0 / 150,132 / 150,147 / 150
6,pick up the orange juice and place it in the basket,0 / 150,41 / 150,54 / 150
7,pick up the butter and place it in the basket,0 / 150,147 / 150,149 / 150
8,pick up the salad dressing and place it in the basket,0 / 150,131 / 150,150 / 150


## 4 · What happened to the named object
**This step:** the outcome of every rollout judged by what the gripper held: the named object was delivered (`success`), another object was grasped first (`wrong object`), nothing was held (`no grasp`), or the named object was held but did not end in the basket (`not delivered`); then, for the two configurations with a plan, how far the rollouts got. Pooled over the seeds.

**Status:** real.

In [5]:
for label, suite in L.SUITES.items():
    print(label)
    display(L.outcome_counts(pooled, suite))
stages = {}
for name in (MAP, ORACLE):
    for label, suite in L.SUITES.items():
        parts = [L.stage_counts(PRO / CONFIGS[name].format(seed), suite) for seed in SEEDS
                 if len(runs[name][seed][suite])]
        stages[f"{name}: {label.split(':')[0]}"] = sum(parts[1:], parts[0]) if parts else pd.Series(dtype=int)
display(pd.DataFrame(stages))

position: the target swaps places with another object


,wrong object,no grasp,rollouts,success,not delivered
"released policy, no plan",1458,42,1500,0,0
"ten-task head of 13 Nov + map plans, kept",108,200,1500,989,203
ten-task head of 13 Nov + ORACLE plan,0,11,1500,1380,109


task: the instruction and the goal name another object


,wrong object,success,no grasp,not delivered,rollouts
"released policy, no plan",1308,130,59,3,1500
"ten-task head of 13 Nov + map plans, kept",30,1022,317,131,1500
ten-task head of 13 Nov + ORACLE plan,2,1129,296,73,1500


,"ten-task head of 13 Nov + map plans, kept: position","ten-task head of 13 Nov + map plans, kept: task",ten-task head of 13 Nov + ORACLE plan: position,ten-task head of 13 Nov + ORACLE plan: task
rollouts,1500,1500,1500,1500
gripper came within 2 cm of the planned grasp point,1401,1216,1495,1277
object knocked over before it was held,63,56,28,93
object held,1193,1153,1489,1202
object in the basket at the end,990,1022,1380,1129
"held, then let go outside the basket",76,44,49,38
"held, still in the gripper at the end",128,87,60,35


## What to take from this

**What this run shows**
- **The result of 13 Nov holds on all 50 start states and over three seeds.** Without any oracle plan, the ten-task head with kept plans of the map head solves 0.660 of the position perturbation and 0.681 of the task perturbation (1500 rollouts each), against 0.67 and 0.68 on 20 start states. The spread over seeds is about one point (standard deviation 0.012 for both), so differences of a few points between configurations of this size are real; the earlier runs with one seed and 200 rollouts could not show that.
- **The released policy does not cope with either perturbation:** 0 of 1500 with the target moved, where it grasps another object in 1458 rollouts, and 0.090 with another object asked for, all of it from one task (alphabet soup, 135 of 150), where the object asked for is the one the policy takes in that layout anyway.
- **The oracle plan bounds what a better plan head could add:** 0.920 and 0.753. Under the position perturbation the gap between kept map plans and the oracle plan is the plan head's: with the oracle plan nearly every task is solved (seven of ten tasks at 141 of 150 or more), while the map plans lose most on butter (71 against 150), bbq sauce (78 against 149), orange juice (73 against 121), ketchup (56 against 104) and tomato sauce (92 against 142), and the gripper grasps another object first in 108 rollouts (0 with the oracle plan).
- **Under the task perturbation the gap is in the action head.** Three tasks fail even with the oracle plan: ketchup (7 with map plans, 31 with the oracle plan, of 150), milk (24 and 24) and orange juice (41 and 54). In these rollouts the gripper often does not even come within 2 cm of the grasp point the oracle plan names (it does in 1277 of 1500 rollouts under this perturbation, against 1495 under the other); most failures are "no grasp" (296 of 371 with the oracle plan).
- **Next to the published numbers for this suite:** 0.66 and 0.68 lie above GAM (0.47, 0.50) and CounterAlign (0.51, 0.26) on both perturbations, and below π0.5 with a test-time affordance field on the position perturbation (0.73; it reports no task-perturbation number).

**What it does not show**
- **A like-for-like comparison of methods.** The base models differ: ours is the released starVLA checkpoint (Qwen3-VL-4B), the others build on π0.5 or are not a VLA. GAM uses the simulator's depth and the affordance-field method depth and GPT-4o at test time; our plan head uses the simulator's camera calibration. Our own released policy scores 0.00 under the position perturbation, where π0.5 is reported at 0.17–0.19.
- **The variation over training.** The seeds vary the policy's sampling noise and the simulator, not the training: there is one trained action head and one plan head per configuration.
- **The other three suites** (Spatial, Goal, LIBERO-10) and the object-appearance and language perturbations of LIBERO-PRO.

**Next**
1. Why the action head fails on ketchup, milk and orange juice under the task perturbation even with the oracle plan: where those objects stand in these scenes, and what the gripper does instead.
2. The plan head under the position perturbation: train it also on closed-loop frames, and try a finer map (more cells, or the wrist camera).
3. Variation over training: retrain the action head and the plan head with two more seeds.
4. The other three suites, which need a plan for skills other than pick-and-place; a second architecture; the full fine-tuning comparison agreed on 12 Nov.